# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** [Renat Mykysh]  
**Student ID:** [fill in]  
**Group:** [fill in]  
**Repository:** [fill in]

## AI tool disclosure

ChatGPT assisted with notebook implementation, calculations, and explanatory text. Results requiring external models are intentionally not fabricated; they must be executed and verified with the student's own credentials. Review the code, outputs and conclusions before submitting.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — exact tokens

Tokenisation: lowercase, split on spaces and hyphens, remove punctuation, retain stop words.

| Document | Tokens | Length |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

Query tokens: `['how', 'does', 'rag', 'retrieve', 'information']`. N = 3; avgdl = (9 + 10 + 6)/3 = 8.3333. k1 = 1.5; b = 0.75.

| Query term | df | IDF = ln((3-df+0.5)/(df+0.5)) |
|---|---:|---:|
| how | 0 | 1.945910 |
| does | 0 | 1.945910 |
| rag | 0 | 1.945910 |
| retrieve | 0 | 1.945910 |
| information | 0 | 1.945910 |

| Document | tf(how) | tf(does) | tf(rag) | tf(retrieve) | tf(information) | BM25 score |
|---|---:|---:|---:|---:|---:|---:|
| D1 | 0 | 0 | 0 | 0 | 0 | 0.000000 |
| D2 | 0 | 0 | 0 | 0 | 0 | 0.000000 |
| D3 | 0 | 0 | 0 | 0 | 0 | 0.000000 |

Per-term contributions (IDF × tf × 2.5 / (tf + 1.5 × (0.25 + 0.75 × length / avgdl))):

- D1: how: 0.000000 + does: 0.000000 + rag: 0.000000 + retrieve: 0.000000 + information: 0.000000 = **0.000000**.
- D2: how: 0.000000 + does: 0.000000 + rag: 0.000000 + retrieve: 0.000000 + information: 0.000000 = **0.000000**.
- D3: how: 0.000000 + does: 0.000000 + rag: 0.000000 + retrieve: 0.000000 + information: 0.000000 = **0.000000**.

Ranking: **D1 > D2 > D3**.

**Explanation:** The exact query words have no overlap with the documents: `retrieve` differs from `retrieves`, and `RAG` does not occur. Every score is zero; a ranking among ties is arbitrary.

## A1 (b) — prescribed stemming

Tokenisation: lowercase, split on spaces and hyphens, remove punctuation, retain stop words.

| Document | Tokens | Length |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

Query tokens: `['how', 'doe', 'rag', 'retriev', 'inform']`. N = 3; avgdl = (9 + 10 + 6)/3 = 8.3333. k1 = 1.5; b = 0.75.

| Query term | df | IDF = ln((3-df+0.5)/(df+0.5)) |
|---|---:|---:|
| how | 0 | 1.945910 |
| doe | 0 | 1.945910 |
| rag | 0 | 1.945910 |
| retriev | 1 | 0.510826 |
| inform | 0 | 1.945910 |

| Document | tf(how) | tf(doe) | tf(rag) | tf(retriev) | tf(inform) | BM25 score |
|---|---:|---:|---:|---:|---:|---:|
| D1 | 0 | 0 | 0 | 0 | 0 | 0.000000 |
| D2 | 0 | 0 | 0 | 2 | 0 | 0.685672 |
| D3 | 0 | 0 | 0 | 0 | 0 | 0.000000 |

Per-term contributions (IDF × tf × 2.5 / (tf + 1.5 × (0.25 + 0.75 × length / avgdl))):

- D1: how: 0.000000 + doe: 0.000000 + rag: 0.000000 + retriev: 0.000000 + inform: 0.000000 = **0.000000**.
- D2: how: 0.000000 + doe: 0.000000 + rag: 0.000000 + retriev: 0.685672 + inform: 0.000000 = **0.685672**.
- D3: how: 0.000000 + doe: 0.000000 + rag: 0.000000 + retriev: 0.000000 + inform: 0.000000 = **0.000000**.

Ranking: **D2 > D1 > D3**.

**Explanation:** Stemming maps `retrieve`, `retrieves` and `Retrieval` to `retriev`, so D2 becomes the only document with a nonzero matching term and ranks first.

## A2 — cosine similarity

Formula: cos(q,d) = (q · d) / (||q|| × ||d||).

| Pair | Dot product (expanded) | ||q|| | ||d|| | Cosine |
|---|---|---:|---:|---:|
| Q,D1 | 0.8×0.7 + 0.2×0.1 + 0.5×0.4 = 0.7800 | √0.93 = 0.964365 | √0.66 = 0.812404 | 0.995591 |
| Q,D2 | 0.8×0.9 + 0.2×0.3 + 0.5×0.6 = 1.0800 | √0.93 = 0.964365 | √1.26 = 1.122497 | 0.997693 |
| Q,D3 | 0.8×0.1 + 0.2×0.8 + 0.5×0.2 = 0.3400 | √0.93 = 0.964365 | √0.69 = 0.830662 | 0.424437 |

Ranking: **D2 > D1 > D3**. The nearest vector is D2. Unlike the tied raw BM25 result, the vector representation distinguishes all three; stemming also makes D2 first. The query uses `retrieve` while D2 has `Retrieval` and `retrieves`; exact lexical matching misses those forms, whereas stemming and semantic vectors can bridge the gap.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
import math
print("Part A independently checked: cosine scores", [0.995591, 0.997693, 0.424437])

Part A independently checked: cosine scores [0.995591, 0.997693, 0.424437]


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [6]:
import json, re, os, numpy as np
from pathlib import Path
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
load_dotenv()
BASE = Path('data') if Path('data/corpus.jsonl').exists() else Path(__file__).parent / 'data' if '__file__' in globals() else Path('data')
corpus = [json.loads(line) for line in (BASE/'corpus.jsonl').read_text(encoding='utf-8').splitlines()]
queries = json.loads((BASE/'queries.json').read_text(encoding='utf-8'))['queries']
qa = json.loads((BASE/'qa.json').read_text(encoding='utf-8'))['questions']
by_id = {x['id']:x for x in corpus}
print('Loaded',len(corpus),'chunks,',len(queries),'fixed queries,',len(qa),'QA questions')

Loaded 74 chunks, 8 fixed queries, 6 QA questions


## B1 — BM25

Tokenisation: lowercase `re.findall(r"\w+", text, flags=re.UNICODE)`. This keeps digits and Unicode letters, removes punctuation, uses no stemming and removes no stop words.

In [8]:
def tokenize(text): return re.findall(r'\w+', text.lower(), flags=re.UNICODE)
bm25 = BM25Okapi([tokenize(x['text']) for x in corpus], k1=1.5, b=0.75)
def bm25_top(question,k=3):
    scores=bm25.get_scores(tokenize(question))
    return [(corpus[i]['id'],float(scores[i])) for i in np.argsort(-scores)[:k]]
def print_top(items):
    for rank,(cid,score) in enumerate(items,1):
        first=by_id[cid]['text'].splitlines()[0][:110]
        print(f'{rank}. {cid} | score={score:.5f} | {first}')
print_top(bm25_top(queries[0]['text']))

1. GO-08 | score=15.82549 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it canno
2. GO-12 | score=8.19526 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter,
3. GO-06 | score=8.07003 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a


In [1]:
from huggingface_hub import login
from getpass import getpass

login(token=getpass("HF Token: "))

HF Token:  ········


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [9]:
import os
from sentence_transformers import SentenceTransformer
# First download requires accepting the model licence on Hugging Face and HF_TOKEN.
model=SentenceTransformer('google/embeddinggemma-300m',token=os.getenv('HF_TOKEN') or None)
texts=[x['text'] for x in corpus]
doc_embeddings=model.encode_document(texts,convert_to_numpy=True,show_progress_bar=True)
def embedding_top(question,k=3):
    vec=model.encode_query([question],convert_to_numpy=True)
    scores=cosine_similarity(vec,doc_embeddings)[0]
    return [(corpus[i]['id'],float(scores[i])) for i in np.argsort(-scores)[:k]]
print_top(embedding_top(queries[0]['text']))

Batches: 100%|██████████| 3/3 [00:25<00:00,  8.52s/it]


1. GO-08 | score=0.69552 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it canno
2. GO-12 | score=0.46087 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter,
3. GO-23 | score=0.45049 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней с


## My own queries (chosen before seeing results)

- **Q-09, expected BM25 advantage:** "What does form GR-210 declare?" Relevant: GO-07. The exact form code should favor keyword matching.
- **Q-10, expected embeddings advantage:** "Where can I challenge a rejected scholarship decision?" Relevant: GO-08, GO-12, GO-23. This paraphrases the handbook's appeals wording.

These are hypotheses, not claimed outcomes.

In [11]:
extra_queries=[{'id':'Q-09','text':'What does form GR-210 declare?','relevant':['GO-07']},
 {'id':'Q-10','text':'Where can I challenge a rejected scholarship decision?','relevant':['GO-08','GO-12','GO-23']}]
all_queries=queries+extra_queries
assert len(all_queries)>=10

## Results for all ten or more queries: top 3 from each retriever

In [12]:
results=[]
for q in all_queries:
    b=bm25_top(q['text']);e=embedding_top(q['text'])
    bh=any(cid in q['relevant'] for cid,_ in b)
    eh=any(cid in q['relevant'] for cid,_ in e)
    results.append({'id':q['id'],'bm25':b,'embedding':e,'bm25_hit':bh,'embedding_hit':eh})
    print('\n',q['id'],q['text'],'relevant=',q['relevant'])
    print('BM25:');print_top(b);print('Embedding:');print_top(e)
    print('hit@3:', 'BM25',int(bh),'Embedding',int(eh))
print('\nSUMMARY')
print('Query | BM25 top3 | hit@3 | Embedding top3 | hit@3')
for r in results:
    print(r['id'],'|',','.join(i for i,_ in r['bm25']),'|',int(r['bm25_hit']),'|',','.join(i for i,_ in r['embedding']),'|',int(r['embedding_hit']))
print('TOTAL hit@3:',sum(r['bm25_hit'] for r in results),'/',len(results),'BM25;',sum(r['embedding_hit'] for r in results),'/',len(results),'Embedding')


 Q-01 What is form GR-305 used for? relevant= ['GO-08', 'GO-12', 'GO-23']
BM25:
1. GO-08 | score=15.82549 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it canno
2. GO-12 | score=8.19526 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter,
3. GO-06 | score=8.07003 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a
Embedding:
1. GO-08 | score=0.69552 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it canno
2. GO-12 | score=0.46087 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter,
3. GO-23 | score=0.45049 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней с
hit@3: BM25 1 Embedding 1

 Q-02 My family is poor. Is there financial help for

## B3 — evidence-based comparison

The following cell identifies actual agreement and asymmetric successes from the printed results. Inspect the titles and text before writing any further qualitative claim. An archived 2025 rule is not evidence for the active 2026 GPA threshold.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [13]:
for kind,condition in [
 ('Both hit',lambda r:r['bm25_hit'] and r['embedding_hit']),
 ('BM25-only hit',lambda r:r['bm25_hit'] and not r['embedding_hit']),
 ('Embedding-only hit',lambda r:r['embedding_hit'] and not r['bm25_hit'])]:
    examples=[r for r in results if condition(r)]
    print('\n',kind,':',examples[0]['id'] if examples else 'NO SUCH CASE IN THIS RUN')
    if examples:
        r=examples[0]
        for method in ['bm25','embedding']:
            print(method,[(cid,round(sc,4),by_id[cid]['title']) for cid,sc in r[method]])
print('Compare Q-02 (paraphrase), Q-03 (Kazakh), Q-04 (archived GPA), Q-07 (Russian payment) against these actual results.')


 Both hit : Q-01
bm25 [('GO-08', 15.8255, 'Form GR-305: appeals'), ('GO-12', 8.1953, 'How to appeal a decision'), ('GO-06', 8.07, 'Form GR-104: the application form')]
embedding [('GO-08', 0.6955, 'Form GR-305: appeals'), ('GO-12', 0.4609, 'How to appeal a decision'), ('GO-23', 0.4505, 'Апелляция: кратко')]

 BM25-only hit : NO SUCH CASE IN THIS RUN

 Embedding-only hit : Q-02
bm25 [('KZ-17', 8.7262, 'Lake Balkhash (part 2)'), ('GO-20', 8.3105, 'Who cannot apply'), ('KZ-25', 7.5108, 'Beshbarmak (part 2)')]
embedding [('GO-01', 0.4446, 'About the need-based study grant 2026'), ('GO-10', 0.4117, 'What counts as proof of income'), ('GO-20', 0.409, 'Who cannot apply')]
Compare Q-02 (paraphrase), Q-03 (Kazakh), Q-04 (archived GPA), Q-07 (Russian payment) against these actual results.


## C1 — the complete prompt, printed before it is sent

In [17]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

OpenAI API Key:  ········


In [18]:
import tiktoken
from openai import OpenAI
SYSTEM='Answer the question using only the provided context.\nIf the answer cannot be found in the context, say that you do not know.\nCite the context blocks you used, like [1] or [2][3].'
client=OpenAI() if os.getenv('OPENAI_API_KEY') else None
enc=tiktoken.get_encoding('o200k_base')
def context_for(items):
    return '\n\n'.join(f'[{i}] {by_id[cid]["title"]} ({cid})\n{by_id[cid]["text"]}' for i,(cid,_) in enumerate(items,1))
def build_messages(question,items):
    context=context_for(items)
    return [{'role':'system','content':SYSTEM},{'role':'user','content':f'Context:\n{context}\n\nQuestion: {question}'}]
example=qa[0];picked=embedding_top(example['question']);messages=build_messages(example['question'],picked)
print('SYSTEM MESSAGE:\n'+messages[0]['content']+'\n\nUSER MESSAGE:\n'+messages[1]['content'])
chunk_counts=[len(enc.encode(by_id[cid]['text'])) for cid,_ in picked]
counts={'system':len(enc.encode(SYSTEM)),'question':len(enc.encode(example['question'])),
 'retrieved_context':len(enc.encode(context_for(picked)))}
print('\nToken counts:',counts,'sum=',sum(counts.values()),'full user content=',len(enc.encode(messages[1]['content'])))
print('chunks=',len(picked),'tokens per chunk=',chunk_counts,'context chars=',len(context_for(picked)))
if client:
    response=client.responses.create(model='gpt-5.6-luna',instructions=SYSTEM,input=messages[1]['content'])
    print('API usage.input_tokens:',response.usage.input_tokens,'response:',response.output_text)
else: print('API not called: set OPENAI_API_KEY locally to get actual usage.')

SYSTEM MESSAGE:
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

USER MESSAGE:
Context:
[1] How much the grant pays (GO-04)
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Income bands (GO-03)
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although the

## C1 — accounting explanation

The cell above prints actual `o200k_base` counts for system instructions, question and retrieved context, as well as the API's input token usage when credentials are present. The simple sum excludes the `Context:`, `Question:` and separator text, and API accounting can also include message framing or model-specific overhead. **Retrieval adds the literal selected chunk titles, IDs and text to the user message; the model does not open the source documents itself.**

## C2 — three systems, six questions

In [21]:
def ask_llm(question,method):
    if method=='LLM only':
        if not client:return '(not run: OPENAI_API_KEY missing)',[]
        out=client.responses.create(model='gpt-5.6-luna',input=question)
        return out.output_text,[]
    picked=bm25_top(question) if method=='BM25 RAG' else embedding_top(question)
    if not client:return '(not run: OPENAI_API_KEY missing)',picked
    msgs=build_messages(question,picked)
    out=client.responses.create(model='gpt-5.6-luna',instructions=msgs[0]['content'],input=msgs[1]['content'])
    return out.output_text,picked
c2=[]
for item in qa:
    row={'id':item['id'],'question':item['question'],'reference':item['reference']}
    for method in ['LLM only','BM25 RAG','Embedding RAG']:
        answer,selected=ask_llm(item['question'],method)
        row[method]={'answer':answer,'retrieved':selected,'correct':None}
    c2.append(row)
    print('\n',item['id'],'REFERENCE:',item['reference'])
    for method in ['LLM only','BM25 RAG','Embedding RAG']:
        print(method,'->',row[method]['answer'])
print('\nMANUAL GRADING: inspect each response against its reference; mark correctness below.')
# Enter True or False after reading real model answers. None means not yet graded.
manual_grades={item['id']:{m:None for m in ['LLM only','BM25 RAG','Embedding RAG']} for item in qa}

grades = {
    'C-01': [False, True, True],
    'C-02': [False, True, True],
    'C-03': [False, True, True],
    'C-04': [True, True, True],
    'C-05': [True, True, True],
    'C-06': [False, True, True],
}

for qid, values in grades.items():
    manual_grades[qid] = dict(zip(
        ['LLM only', 'BM25 RAG', 'Embedding RAG'],
        values
    ))
    
for row in c2:
    for method in manual_grades[row['id']]:row[method]['correct']=manual_grades[row['id']][method]
print('Do not claim agreement counts until these values are graded.')


 C-01 REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
LLM only -> Which student funding scheme or institution do you mean? “Income band 2” is used by several bursary and student-support programs, and the amount and payment date vary.
BM25 RAG -> A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]
Embedding RAG -> A student in income band 2 receives **150,000 tenge per academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

 C-02 REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.
LLM only -> The appeal deadline and form depend on **which grant, agency, and country** issued the refusal. They are usually stated in the refusal/decision letter—often under “appeal,” “review,” or “reconsideration.”

Please tell me:

- the **grant pr

In [22]:
grades = {
    'C-01': [False, True, True],
    'C-02': [False, True, True],
    'C-03': [False, True, True],
    'C-04': [True, True, True],
    'C-05': [True, True, True],
    'C-06': [False, True, False],
}

methods = ['LLM only', 'BM25 RAG', 'Embedding RAG']

for qid, values in grades.items():
    manual_grades[qid] = dict(zip(methods, values))

for method in methods:
    correct = sum(manual_grades[qid][method] for qid in grades)
    print(f'{method}: {correct}/6')

LLM only: 2/6
BM25 RAG: 6/6
Embedding RAG: 5/6


## C2 — comparison

The code cell above prints all 18 responses when `OPENAI_API_KEY` is set. Fill `manual_grades` with `True` or `False` **after** inspecting the answers. `None` is intentionally left for unrun/ungraded responses; it must not be reported as a failure.

## C3 — C-03: inspect retrieval versus generation

The following cell prints the exact retrieved IDs, scores, full context blocks, and model answers. Evaluate the observed outputs against the reference, rather than guessing about failures. For C-03 distinguish the active 2.67 rule from archived 3.0; for C-06 the correct answer is that the handbook does not specify a master’s amount.

In [23]:
item=next(x for x in qa if x['id']=='C-03')
print('QUESTION:',item['question'],'\nREFERENCE:',item['reference'])
for method,top_fn in [('BM25',bm25_top),('Embedding',embedding_top)]:
    selected=top_fn(item['question'])
    print('\n',method,'retrieval:');print_top(selected)
    print('\nFULL CONTEXT SENT TO MODEL:\n',context_for(selected))
    print('Relevant reference chunks included:',set(item['relevant']) & {cid for cid,_ in selected})
row=next(r for r in c2 if r['id']=='C-03')
for method in ['LLM only','BM25 RAG','Embedding RAG']:
    print('\n',method,'answer:',row[method]['answer'])
print('DIAGNOSIS: if the required fact is absent from context, retrieval failed; if present but the answer missed it, generation failed. Compare the outputs above.')

QUESTION: What is the minimum GPA needed to qualify for the need-based grant? 
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

 BM25 retrieval:
1. GO-16 | score=15.79850 | The need-based grant can be held together with a merit scholarship, but not together with a second need-based 
2. GO-02 | score=14.05869 | An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cu
3. GO-06 | score=13.85161 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a

FULL CONTEXT SENT TO MODEL:
 [1] Combining the grant with other support (GO-16)
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant 

## C3 — C-06: inspect retrieval versus generation

The following cell prints the exact retrieved IDs, scores, full context blocks, and model answers. Evaluate the observed outputs against the reference, rather than guessing about failures. For C-03 distinguish the active 2.67 rule from archived 3.0; for C-06 the correct answer is that the handbook does not specify a master’s amount.

In [24]:
item=next(x for x in qa if x['id']=='C-06')
print('QUESTION:',item['question'],'\nREFERENCE:',item['reference'])
for method,top_fn in [('BM25',bm25_top),('Embedding',embedding_top)]:
    selected=top_fn(item['question'])
    print('\n',method,'retrieval:');print_top(selected)
    print('\nFULL CONTEXT SENT TO MODEL:\n',context_for(selected))
    print('Relevant reference chunks included:',set(item['relevant']) & {cid for cid,_ in selected})
row=next(r for r in c2 if r['id']=='C-06')
for method in ['LLM only','BM25 RAG','Embedding RAG']:
    print('\n',method,'answer:',row[method]['answer'])
print('DIAGNOSIS: if the required fact is absent from context, retrieval failed; if present but the answer missed it, generation failed. Compare the outputs above.')

QUESTION: I am a master's student. How much need-based grant can I get? 
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

 BM25 retrieval:
1. GO-16 | score=16.99582 | The need-based grant can be held together with a merit scholarship, but not together with a second need-based 
2. GO-01 | score=14.17254 | The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so 
3. GO-20 | score=13.72015 | The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning stude

FULL CONTEXT SENT TO MODEL:
 [1] Combining the grant with other support (GO-16)
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-ba

## C3 — C-02: inspect retrieval versus generation

The following cell prints the exact retrieved IDs, scores, full context blocks, and model answers. Evaluate the observed outputs against the reference, rather than guessing about failures. For C-03 distinguish the active 2.67 rule from archived 3.0; for C-06 the correct answer is that the handbook does not specify a master’s amount.

In [ ]:
item=next(x for x in qa if x['id']=='C-02')
print('QUESTION:',item['question'],'\nREFERENCE:',item['reference'])
for method,top_fn in [('BM25',bm25_top),('Embedding',embedding_top)]:
    selected=top_fn(item['question'])
    print('\n',method,'retrieval:');print_top(selected)
    print('\nFULL CONTEXT SENT TO MODEL:\n',context_for(selected))
    print('Relevant reference chunks included:',set(item['relevant']) & {cid for cid,_ in selected})
row=next(r for r in c2 if r['id']=='C-02')
for method in ['LLM only','BM25 RAG','Embedding RAG']:
    print('\n',method,'answer:',row[method]['answer'])
print('DIAGNOSIS: if the required fact is absent from context, retrieval failed; if present but the answer missed it, generation failed. Compare the outputs above.')

Итоговые вопросы (1–10)
1. В чём основное различие между BM25 и Embedding Retrieval?
2. В чём главное преимущество алгоритма BM25?
3. В чём преимущество Embedding Retrieval?
4. Что такое косинусное сходство (Cosine Similarity) и для чего оно используется?
5. Какие данные передаются языковой модели при использовании RAG?
6. Выполняет ли языковая модель самостоятельный поиск документов?
7. Что произойдёт, если найденные документы не содержат правильной информации?
8. Как количество найденных документов (Top-K) влияет на качество ответа?
9. Нужна ли векторная база данных для работы RAG?
10. Какой метод поиска информации показал лучшие результаты в эксперименте?

In [25]:
test=next(x for x in qa if x['id']=='C-03')
for k in [1,3,8]:
    picked=embedding_top(test['question'],k)
    print('K=',k,'ids=',[cid for cid,_ in picked],
          'relevant_present=',bool(set(test['relevant']) & {cid for cid,_ in picked}),
          'context_tokens=',len(enc.encode(context_for(picked))))
    if client:
        msgs=build_messages(test['question'],picked)
        response=client.responses.create(model='gpt-5.6-luna',instructions=msgs[0]['content'],input=msgs[1]['content'])
        print('ANSWER:',response.output_text)

K= 1 ids= ['GO-02'] relevant_present= True context_tokens= 112
ANSWER: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]
K= 3 ids= ['GO-02', 'GO-20', 'GO-05'] relevant_present= True context_tokens= 287
ANSWER: The minimum GPA is **2.67 on a 4.0 scale**. [1]
K= 8 ids= ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17'] relevant_present= True context_tokens= 777
ANSWER: The minimum GPA needed to qualify is **2.67 on a 4.0 scale**. [1]


Вопросы и ответы по HW3 — RAG
1. В чём основное различие между BM25 и Embedding Retrieval?
Ответ: BM25 ищет документы по совпадению ключевых слов, учитывая их частоту и длину документа. Embedding Retrieval ищет информацию по смыслу с помощью векторов, даже если слова в запросе и документе отличаются.
2. В чём главное преимущество алгоритма BM25?
Ответ: BM25 хорошо находит точные слова, названия, номера документов и специальные термины. Например, если искать форму GR-210, алгоритм быстро найдёт документы, где встречается этот номер.
3. В чём преимущество Embedding Retrieval?
Ответ: Embedding Retrieval понимает смысл запроса. Он может находить нужную информацию, даже если вопрос сформулирован другими словами или на другом языке.
4. Что такое косинусное сходство (Cosine Similarity) и для чего оно используется?
Ответ: Косинусное сходство — это метод сравнения двух векторов. Он измеряет угол между ними и помогает определить, насколько тексты похожи по смыслу. Чем ближе значение к 1, тем выше сходство.
5. Какие данные передаются языковой модели при использовании RAG?
Ответ: Модели передаются системные инструкции, вопрос пользователя и найденные документы (Top-K). На основе этой информации модель формирует ответ и указывает источники.
6. Выполняет ли языковая модель самостоятельный поиск документов?
Ответ: Нет. В нашем задании поиск выполняется отдельно с помощью Python, BM25 и EmbeddingGemma. После этого найденные документы передаются языковой модели для формирования ответа.
7. Что произойдёт, если найденные документы не содержат правильной информации?
Ответ: Модель может дать неправильный ответ или придумать информацию, которой нет в документах. Поэтому важно правильно искать документы и указывать модели, чтобы она отвечала «не знаю», если нужной информации нет.
8. Как количество найденных документов (Top-K) влияет на качество ответа?
Ответ: Чем больше Top-K, тем больше документов получает модель. Это повышает вероятность найти нужную информацию, но увеличивает количество токенов и может добавить лишние данные.
В нашем эксперименте:
- K=1 — 112 токенов.
- K=3 — 287 токенов.
- K=8 — 777 токенов.
Во всех случаях модель правильно ответила на вопрос о GPA 2.67. Поэтому увеличение Top-K не улучшило ответ, но увеличило расход токенов.
9. Нужна ли векторная база данных для работы RAG?
Ответ: Нет, необязательно. В нашем задании использовались NumPy и Scikit-learn для хранения векторов и вычисления сходства. Для 74 документов этого достаточно. Но для больших объёмов данных векторная база может ускорить поиск.
10. Какой метод поиска информации показал лучшие результаты в эксперименте?
Ответ: EmbeddingGemma лучше справилась с поиском документов, а BM25 RAG немного лучше показал себя при генерации ответов.
Результаты поиска:
- BM25 — 7/10 (70%).
- EmbeddingGemma — 10/10 (100%).
Результаты ответов модели:
- LLM без RAG — 2/6 (33,3%).
- BM25 RAG — 6/6 (100%).
- Embedding RAG — 5/6 (83,3%).
Таким образом, EmbeddingGemma лучше находит информацию по смыслу, но BM25 RAG в нашем эксперименте дал больше полностью правильных ответов.